In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# USGS MCS 2025 (up to 2023, prevision to 2024)

In [60]:
df_usgs_mcs_raw = pd.read_csv(r'data/data_minerals/USGS/Mineral_commodity_summary_2025/MCS2025_World_Data.csv')

In [61]:
def restructure_mcs(df):
    # Step 1: Keep only rows where COUNTRY contains 'World'
    df_world = df[df['COUNTRY'].str.contains("World", case=False, na=False)].copy()
    df_world['COUNTRY'] = 'World total'

    # Step 2: Standardize column names for easier parsing
    df_world.rename(columns={
        'PROD_2023': '2023_PROD',
        'PROD_EST_ 2024': '2024_PROD',
        'PROD_NOTES': 'PROD_NOTES',
        'CAP_2023': '2023_CAP',
        'CAP_EST_ 2024': '2024_CAP',
        'CAP_NOTES': 'CAP_NOTES',
        'RESERVES_2024': '2024_RES',
        'RESERVE_NOTES': 'RES_NOTES'
    }, inplace=True)

    # Step 3: Melt the production, capacity, and reserves values
    value_df = pd.melt(
        df_world,
        id_vars=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS'],
        value_vars=['2023_PROD', '2024_PROD', '2023_CAP', '2024_CAP', '2024_RES'],
        var_name='YEAR_DATATYPE',
        value_name='VALUE'
    )

    # Step 4: Extract YEAR and DATA_TYPE
    value_df[['YEAR', 'DATA_TYPE']] = value_df['YEAR_DATATYPE'].str.extract(r'(\d{4})_(PROD|CAP|RES)')
    value_df['DATA_TYPE'] = value_df['DATA_TYPE'].map({'PROD': 'Production', 'CAP': 'Capacity', 'RES': 'Reserves'})

    # Step 5: Melt the notes in long format and map their types
    note_df = pd.melt(
        df_world,
        id_vars=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS'],
        value_vars=['PROD_NOTES', 'CAP_NOTES', 'RES_NOTES'],
        var_name='NOTE_COLUMN',
        value_name='NOTES'
    )
    note_df['DATA_TYPE'] = note_df['NOTE_COLUMN'].str.extract(r'(PROD|CAP|RES)')[0].map({'PROD': 'Production', 'CAP': 'Capacity', 'RES': 'Reserves'})
    note_df.drop(columns='NOTE_COLUMN', inplace=True)

    # Step 6: Merge values with corresponding notes
    merged_df = pd.merge(
        value_df,
        note_df,
        on=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS', 'DATA_TYPE'],
        how='left'
    )

    # Step 7: Drop rows where VALUE is NaN and sort
    cleaned_df = merged_df.dropna(subset=['VALUE']).sort_values(by='COMMODITY').reset_index(drop=True)

    # Step 8: Return only relevant columns
    return cleaned_df[['COMMODITY', 'COUNTRY', 'TYPE', 'DATA_TYPE', 'UNIT_MEAS', 'YEAR', 'VALUE', 'NOTES']]

In [62]:
# Apply the function
df_usgs_mcs = restructure_mcs(df_usgs_mcs_raw)

In [59]:
df_usgs_mcs

,COMMODITY,COUNTRY,TYPE,DATA_TYPE,UNIT_MEAS,YEAR,VALUE,NOTES
0,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2023,1310000.0,NaN
1,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2023,1010000.0,NaN
2,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2024,1310000.0,NaN
3,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2024,1010000.0,NaN
4,Aluminum,World total,"Smelter production, aluminum",Production,thousand metric tons,2023,70000.0,NaN
...,...,...,...,...,...,...,...,...
223,Zinc,World total,"Mine production, zinc content",Production,thousand metric tons,2023,12100.0,NaN
224,Zinc,World total,"Mine production, zinc content",Reserves,thousand metric tons,2024,230000,NaN
225,Zirconium and Hafnium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2023,1440.0,NaN
226,Zirconium and Hafnium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2024,1500.0,NaN


In [65]:
df_usgs_mcs['COMMODITY'] = df_usgs_mcs['COMMODITY'].replace({
    "Zirconium and Hafnium": "Zirconium"})

In [66]:
df_usgs_mcs

,COMMODITY,COUNTRY,TYPE,DATA_TYPE,UNIT_MEAS,YEAR,VALUE,NOTES
0,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2023,1310000.0,NaN
1,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2023,1010000.0,NaN
2,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2024,1310000.0,NaN
3,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2024,1010000.0,NaN
4,Aluminum,World total,"Smelter production, aluminum",Production,thousand metric tons,2023,70000.0,NaN
...,...,...,...,...,...,...,...,...
223,Zinc,World total,"Mine production, zinc content",Production,thousand metric tons,2023,12100.0,NaN
224,Zinc,World total,"Mine production, zinc content",Reserves,thousand metric tons,2024,230000,NaN
225,Zirconium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2023,1440.0,NaN
226,Zirconium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2024,1500.0,NaN


In [74]:
df_usgs_mcs.to_csv(r'data/data_minerals/df_usgs_mcs.csv', index=False)

In [67]:
com_usgs_mcs = df_usgs_mcs['COMMODITY'].str.lower().unique().tolist()

# WMD 2024 (up to 2022)

In [45]:
df_wmd = pd.read_excel(r'data/data_minerals/WMD/6.2. World_Production_of_Mineral_Raw_Materials_by_ Mineral_Raw_Materials.xlsx')

In [68]:
df_wmd['Commodity'] = df_wmd['Commodity'].replace({
    "Aluminium": "Aluminum",
    "Diam. (Gem)": "Diamond (gemstones)",
    "Diam. (Ind)": "Diamond (industrial)",
    "Zircon": "Zirconium",
})
df_wmd

,Commodity,Commodity_group,Unit,2018,2019,2020,2021,2022
0,Aluminum,Non-Ferrous Metals,tons,64443051,63230201,65404146,67049920,68825618
1,Antimony,Non-Ferrous Metals,tons,145002,134659,119911,91920,83031
2,Arsenic,Non-Ferrous Metals,tons,51737,55243,52298,54695,56404
3,Asbestos,Industrial Minerals,tons,1237256,1137505,1132268,1302356,1258065
4,Baryte,Industrial Minerals,tons,9627288,9510043,7841612,8358423,8278342
...,...,...,...,...,...,...,...,...
62,Uranium,Minerals Fuels,tons,64083,64278,55718,55981,55972
63,Vanadium,Iron & Ferro-Alloy Metals,tons,84947,98723,105776,110269,115271
64,Vermiculite,Industrial Minerals,tons,438951,424421,416638,456936,502825
65,Zinc,Non-Ferrous Metals,tons,12252248,12988998,12669413,12843283,12762076


In [69]:
com_wmd = df_wmd['Commodity'].str.lower().unique().tolist()

# Lists of minerals covered in the IEA scenarios

In [70]:
# List of metals included in the IEA scenarios
com_iea = [
    "aluminum",
    "arsenic",
    "boron",
    "cadmium",
    "chromium",
    "cobalt",
    "copper",
    "dysprosium",
    "ferromanganese",
    "ferronickel",
    "gallium",
    "graphite",
    "indium",
    "iridium",
    "lead",
    "lithium",
    "manganese",
    "molybdenum",
    "neodymium",
    "nickel",
    "platinum",
    "praseodymium",
    "selenium",
    "silicon",
    "silver",
    "tellurium",
    "terbium",
    "tin",
    "vanadium",
    "zinc",
    "zirconium"
]

# Comparison

In [71]:
# Make everything lowercase and strip whitespace
com_usgs_mcs = [m.strip().lower() for m in com_usgs_mcs]
com_wmd = [m.strip().lower() for m in com_wmd]
com_iea = [m.strip().lower() for m in com_iea]

unique_minerals = sorted(set(com_usgs_mcs + com_wmd + com_iea))

df_comparison = pd.DataFrame(unique_minerals, columns=["Mineral"])
df_comparison["In_USGS_MCS2024"] = df_comparison["Mineral"].isin(com_usgs_mcs)
df_comparison["In_WMD_2024"] = df_comparison["Mineral"].isin(com_wmd)
df_comparison["In_IEA"] = df_comparison["Mineral"].isin(com_iea)

In [72]:
df_comparison

,Mineral,In_USGS_MCS2024,In_WMD_2024,In_IEA
0,abrasives,True,False,False
1,aluminum,True,True,True
2,antimony,True,True,False
3,arsenic,True,True,True
4,asbestos,True,True,False
...,...,...,...,...
98,vermiculite,True,True,False
99,wollastonite,True,False,False
100,zeolites (natural),True,False,False
101,zinc,True,True,True


In [73]:
df_comparison.to_csv(r'df_comparison.csv', index=False)

In [105]:
missing_metals = [
'dysprosium', # REE # Lundaev et al 2023?
'ferromanganese', # # Part of the manganese file in the Mineral Yearbook
'ferronickel',# # Part of the nickel file in the Mineral Yearbook
'iridium', # PGM 
'neodymium', # REE
'praseodymium', # REE
'terbium' # REE 
]